# Simulation

> Simulate study and free-recall trials from memory-search models.

The `jaxcmr.simulation` module provides functions for running study and free-recall simulations. Trial-level functions take a `MemorySearch` model and return recalled item indices; dataset-level functions wrap these to simulate across many trials with per-subject parameters using `vmap`.

In [1]:
#| include: false
from nbdev.showdoc import show_doc

In [2]:
#| code-summary: Imports
import warnings

import jax.numpy as jnp
from jax import random

from jaxcmr.models.cmr import CMR, make_factory
from jaxcmr.components.linear_memory import init_mfc, init_mcf
from jaxcmr.components.context import init as init_context
from jaxcmr.components.termination import PositionalTermination
from jaxcmr.simulation import (
    item_to_study_positions,
    simulate_free_recall,
    simulate_free_recall_count,
    simulate_study_and_free_recall,
    simulate_study_first_recall_and_free_recall,
    simulate_study_free_recall_and_forced_stop,
    MemorySearchSimulator,
    preallocate_for_h5_dataset,
    simulate_h5_from_h5,
    parameter_shifted_simulate_h5_from_h5,
)

warnings.filterwarnings("ignore")

In [3]:
#| code-summary: Model parameters
params = {
    "encoding_drift_rate": 0.7,
    "start_drift_rate": 0.6,
    "recall_drift_rate": 0.85,
    "learning_rate": 0.4,
    "primacy_scale": 8.0,
    "primacy_decay": 1.0,
    "item_support": 6.0,
    "shared_support": 0.02,
    "choice_sensitivity": 1.5,
    "stop_probability_scale": 0.005,
    "stop_probability_growth": 0.4,
    "learn_after_context_update": True,
    "allow_repeated_recalls": False,
}

---

## Trial-Level Simulation

To simulate a free recall trial, `simulate_study_and_free_recall` encodes a study list into a model and then runs free recall in one call. It follows the `TrialSimulator` protocol: model, presentation sequence, observed recalls, and random key.

In [4]:
#| echo: false
show_doc(simulate_study_and_free_recall)

---

### simulate_study_and_free_recall

>      simulate_study_and_free_recall (model:jaxcmr.typing.MemorySearch,
>                                      present:jaxtyping.Integer[Array,'study_ev
>                                      ents'],
>                                      trial:jaxtyping.Integer[Array,'recalls'],
>                                      rng:Union[jaxtyping.Key[Array,''],jaxtypi
>                                      ng.UInt32[Array,'2']])

*Simulate study then free recall for a single trial.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| model | MemorySearch | Memory search model in study mode. |
| present | Integer[Array, 'study_events'] | One-indexed study sequence for the trial. |
| trial | Integer[Array, 'recalls'] | Observed recall sequence for the trial. |
| rng | Union | Random key. |
| **Returns** | **tuple** | **Updated model and recalled item indices.** |

In [5]:
#| code-summary: Simulate a 16-item free recall trial
model = CMR(list_length=16, parameters=params)
present = jnp.arange(1, 17)  # 1-indexed items
rng = random.PRNGKey(42)

model, recalls = simulate_study_and_free_recall(
    model, present, present, rng
)
recalls

Array([ 1,  2,  4,  3,  7, 16,  6,  8,  5,  0,  0,  0,  0,  0,  0,  0],      dtype=int32, weak_type=True)

Nonzero entries are recalled item indices (1-indexed, matching the presentation sequence). Zeros indicate unused recall slots. The `trial` parameter is part of the `TrialSimulator` interface but is not used by this function—variant functions like `simulate_study_first_recall_and_free_recall` use it to condition simulation on observed data.

---

## Dataset-Level Simulation

To scale simulation to many trials, `simulate_h5_from_h5` takes a model factory and an HDF5-shaped dataset and runs trial-level simulation across all selected trials using `vmap`. Per-subject parameters allow each participant's fitted values to drive their simulated trials.

In [6]:
#| echo: false
show_doc(simulate_h5_from_h5)

---

### simulate_h5_from_h5

>      simulate_h5_from_h5
>                           (model_factory:Type[jaxcmr.typing.MemorySearchModelF
>                           actory], dataset:jaxcmr.typing.RecallDataset, featur
>                           es:Optional[jaxtyping.Float[Array,'word_pool_itemsfe
>                           atures_count']], parameters:dict[str,jaxtyping.Float
>                           [Array,'subject_count']],
>                           trial_mask:jaxtyping.Bool[Array,'trial_count'],
>                           experiment_count:int, rng:Union[jaxtyping.Key[Array,
>                           ''],jaxtyping.UInt32[Array,'2']], size:int=3, simula
>                           te_trial_fn:jaxcmr.typing.TrialSimulator=<function
>                           simulate_study_and_free_recall>)

*Simulate a dataset using per-subject parameters.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model_factory | Type |  | Factory class for memory search models. |
| dataset | RecallDataset |  | Original dataset containing trial data. |
| features | Optional |  | Feature matrix for the word pool, or None. |
| parameters | dict |  | Per-subject simulation parameters. |
| trial_mask | Bool[Array, 'trial_count'] |  | Mask selecting trials to simulate. |
| experiment_count | int |  | Replications per selected trial. |
| rng | Union |  | Random key. |
| size | int | 3 | Max study positions per item when reindexing. |
| simulate_trial_fn | TrialSimulator | simulate_study_and_free_recall | Trial-level simulation function. |
| **Returns** | **RecallDataset** |  | **Simulated dataset with recall sequences.** |

In [7]:
#| code-summary: Build a dataset and simulate
CMRFactory = make_factory(
    init_mfc, init_mcf, init_context, PositionalTermination
)

dataset = {
    "subject": jnp.array([[0], [0]], dtype=jnp.int32),
    "listLength": jnp.full((2, 1), 5, dtype=jnp.int32),
    "pres_itemnos": jnp.tile(jnp.arange(1, 6), (2, 1)),
    "recalls": jnp.zeros((2, 5), dtype=jnp.int32),
}

sim_params = {
    k: jnp.array([v], dtype=jnp.float32)
    for k, v in params.items()
    if isinstance(v, (int, float))
}

mask = jnp.ones(2, dtype=bool)
rng = random.PRNGKey(0)

sim_data = simulate_h5_from_h5(
    CMRFactory, dataset, None, sim_params, mask, 1, rng
)
sim_data["recalls"]

Array([[1, 4, 5, 2, 3],
       [4, 5, 1, 3, 2]], dtype=int32)

---

## Reference

`simulate_free_recall` is the lower-level building block that runs only the recall phase—it assumes the model is already in retrieval mode. `simulate_free_recall_count` adds a cap on the number of recall events.

In [8]:
#| echo: false
show_doc(simulate_free_recall)

---

### simulate_free_recall

>      simulate_free_recall (model:jaxcmr.typing.MemorySearch, list_length:int,
>                            rng:Union[jaxtyping.Key[Array,''],jaxtyping.UInt32[
>                            Array,'2']])

*Simulate repeated free-recall steps from a model.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| model | MemorySearch | Retrieval-ready memory search model. |
| list_length | int | Upper bound on recall steps to simulate. |
| rng | Union | Random key. |
| **Returns** | **tuple** | **Updated model and recalled item indices.** |

In [9]:
#| echo: false
show_doc(simulate_free_recall_count)

---

### simulate_free_recall_count

>      simulate_free_recall_count (model:jaxcmr.typing.MemorySearch,
>                                  recall_count:jaxtyping.Integer[Array,'']|int,
>                                  list_length:int, rng:Union[jaxtyping.Key[Arra
>                                  y,''],jaxtyping.UInt32[Array,'2']])

*Simulate free recall truncated to a fixed event count.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| model | MemorySearch | Retrieval-ready memory search model. |
| recall_count | jaxtyping.Integer[Array, ''] \| int | Maximum retrieval events to simulate. |
| list_length | int | Total recall slots in the output. |
| rng | Union | Random key. |
| **Returns** | **tuple** | **Updated model and recalled item indices.** |

Two variant trial functions handle special experimental designs. `simulate_study_first_recall_and_free_recall` forces the first recalled item to match the observed data, then simulates the rest freely—useful for conditional-recall analyses.

In [10]:
#| echo: false
show_doc(simulate_study_first_recall_and_free_recall)

---

### simulate_study_first_recall_and_free_recall

>      simulate_study_first_recall_and_free_recall
>                                                   (model:jaxcmr.typing.MemoryS
>                                                   earch, present:jaxtyping.Int
>                                                   eger[Array,'study_events'], 
>                                                   trial:jaxtyping.Integer[Arra
>                                                   y,'recalls'], rng:Union[jaxt
>                                                   yping.Key[Array,''],jaxtypin
>                                                   g.UInt32[Array,'2']])

*Simulate study, forced first recall, then free recall.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| model | MemorySearch | Memory search model in study mode. |
| present | Integer[Array, 'study_events'] | One-indexed study sequence for the trial. |
| trial | Integer[Array, 'recalls'] | Observed recall sequence; first element is forced. |
| rng | Union | Random key. |
| **Returns** | **tuple** | **Updated model and recalled item indices.** |

`simulate_study_free_recall_and_forced_stop` runs unconstrained free recall but post-processes the output to match observed stop timing: recall indices are zeroed wherever the observed trial has a zero (i.e., the participant had already stopped).

In [11]:
#| echo: false
show_doc(simulate_study_free_recall_and_forced_stop)

---

### simulate_study_free_recall_and_forced_stop

>      simulate_study_free_recall_and_forced_stop
>                                                  (model:jaxcmr.typing.MemorySe
>                                                  arch, present:jaxtyping.Integ
>                                                  er[Array,'study_events'], tri
>                                                  al:jaxtyping.Integer[Array,'r
>                                                  ecalls'], rng:Union[jaxtyping
>                                                  .Key[Array,''],jaxtyping.UInt
>                                                  32[Array,'2']])

*Simulate study and free recall with stop timing from data.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| model | MemorySearch | Memory search model in study mode. |
| present | Integer[Array, 'study_events'] | One-indexed study sequence for the trial. |
| trial | Integer[Array, 'recalls'] | Observed recall sequence; first zero marks stop. |
| rng | Union | Random key. |
| **Returns** | **tuple** | **Updated model and masked recall indices.** |

`item_to_study_positions` maps a recalled item back to the one-indexed positions where it appeared during study. This is used internally to reindex simulation output into study-position format.

In [12]:
#| echo: false
show_doc(item_to_study_positions)

---

### item_to_study_positions

>      item_to_study_positions (item:jaxtyping.Integer[Array,'']|int,
>                               presentation:jaxtyping.Integer[Array,'list_lengt
>                               h'], size:int)

*Map an item to its one-indexed study positions.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| item | jaxtyping.Integer[Array, ''] \| int | Item index (0 means no item). |
| presentation | Integer[Array, 'list_length'] | Presentation sequence for a single trial. |
| size | int | Number of positions to return. |
| **Returns** | **Integer[Array, 'size']** | **One-indexed study positions, zero-padded.** |

In [13]:
present = jnp.array([5, 7, 5, 5])
out = item_to_study_positions(5, present, size=3)
assert out.tolist() == [1, 3, 4]

# Padding item (0) always maps to zeros
out = item_to_study_positions(0, present, size=3)
assert jnp.all(out == 0)

`preallocate_for_h5_dataset` selects trials by mask and replicates them for multi-simulation runs.

In [14]:
#| echo: false
show_doc(preallocate_for_h5_dataset)

---

### preallocate_for_h5_dataset

>      preallocate_for_h5_dataset (data:jaxcmr.typing.RecallDataset,
>                                  trial_mask:jaxtyping.Bool[Array,'trial_count'
>                                  ], experiment_count:int)

*Replicate selected trials for batch simulation.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| data | RecallDataset | Input dataset. |
| trial_mask | Bool[Array, 'trial_count'] | Mask selecting trials to include. |
| experiment_count | int | Replication count per trial. |
| **Returns** | **RecallDataset** | **Dataset with selected trials replicated.** |

In [15]:
data = {
    "subject": jnp.array([[0], [1]]),
    "pres_itemnos": jnp.array([[1, 2, 3], [4, 5, 6]]),
    "recalls": jnp.array([[1, 0, 0], [2, 0, 0]]),
    "listLength": jnp.array([[3], [3]]),
}
mask = jnp.array([True, False])

out = preallocate_for_h5_dataset(data, mask, experiment_count=2)
assert out["subject"].shape[0] == 2
assert jnp.all(
    out["pres_itemnos"] == jnp.array([[1, 2, 3], [1, 2, 3]])
)

`MemorySearchSimulator` wraps a model factory and dataset into a stateless object whose `simulate_trial` method can be `vmap`-ed over trials. It is used internally by `simulate_h5_from_h5`.

In [16]:
#| echo: false
show_doc(MemorySearchSimulator.__init__)

---

### MemorySearchSimulator.__init__

>      MemorySearchSimulator.__init__
>                                      (model_factory:Type[jaxcmr.typing.MemoryS
>                                      earchModelFactory],
>                                      dataset:jaxcmr.typing.RecallDataset, feat
>                                      ures:Optional[jaxtyping.Float[Array,'word
>                                      _pool_itemsfeatures_count']], simulate_tr
>                                      ial_fn:jaxcmr.typing.TrialSimulator=<func
>                                      tion simulate_study_and_free_recall>)

*Initialize simulator from a dataset and model factory.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model_factory | Type |  | Factory class for memory search models. |
| dataset | RecallDataset |  | Dataset with presentation and recall sequences. |
| features | Optional |  | Feature matrix for the word pool, or None. |
| simulate_trial_fn | TrialSimulator | simulate_study_and_free_recall | Trial simulation function. |
| **Returns** | **None** |  |  |

In [17]:
#| echo: false
show_doc(MemorySearchSimulator.simulate_trial)

---

### MemorySearchSimulator.simulate_trial

>      MemorySearchSimulator.simulate_trial
>                                            (trial_index:jaxtyping.Integer[Arra
>                                            y,''], subject_index:jaxtyping.Inte
>                                            ger[Array,'subject_count'], paramet
>                                            ers:Mapping[str,jaxtyping.Float[Arr
>                                            ay,'']|float|int], rng:Union[jaxtyp
>                                            ing.Key[Array,''],jaxtyping.UInt32[
>                                            Array,'2']])

*Simulate a single trial using per-subject parameters.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| trial_index | Integer[Array, ''] | Index of the trial to simulate. |
| subject_index | Integer[Array, 'subject_count'] | Index into per-subject parameter arrays. |
| parameters | Mapping | Per-subject model parameters. |
| rng | Union | Random key. |
| **Returns** | **Integer[Array, 'recall_events']** | **Recalled item indices for the trial.** |

`parameter_shifted_simulate_h5_from_h5` generates multiple simulated datasets by sweeping a single parameter across a set of values, keeping all other parameters fixed.

In [18]:
#| echo: false
show_doc(parameter_shifted_simulate_h5_from_h5)

---

### parameter_shifted_simulate_h5_from_h5

>      parameter_shifted_simulate_h5_from_h5
>                                             (model_factory:Type[jaxcmr.typing.
>                                             MemorySearchModelFactory], dataset
>                                             :jaxcmr.typing.RecallDataset, feat
>                                             ures:Optional[jaxtyping.Float[Arra
>                                             y,'word_pool_itemsfeatures_count']
>                                             ], parameters:dict[str,jaxtyping.F
>                                             loat[Array,'subject_count']], tria
>                                             l_mask:jaxtyping.Bool[Array,'trial
>                                             _count'], experiment_count:int,
>                                             varied_parameter:str,
>                                             parameter_values:Sequence[float], 
>                                             rng:Union[jaxtyping.Key[Array,''],
>                                             jaxtyping.UInt32[Array,'2']],
>                                             size:int=3, simulate_trial_fn:jaxc
>                                             mr.typing.TrialSimulator=<function
>                                             simulate_study_and_free_recall>)

*Simulate multiple datasets by sweeping a single parameter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model_factory | Type |  | Factory class for memory search models. |
| dataset | RecallDataset |  | Original dataset containing trial data. |
| features | Optional |  | Feature matrix for the word pool, or None. |
| parameters | dict |  | Per-subject simulation parameters. |
| trial_mask | Bool[Array, 'trial_count'] |  | Mask selecting trials to simulate. |
| experiment_count | int |  | Replications per selected trial. |
| varied_parameter | str |  | Name of the parameter to sweep. |
| parameter_values | Sequence |  | Values to assign to the swept parameter. |
| rng | Union |  | Random key. |
| size | int | 3 | Max study positions per item when reindexing. |
| simulate_trial_fn | TrialSimulator | simulate_study_and_free_recall | Trial-level simulation function. |
| **Returns** | **Sequence** |  | **One simulated dataset per parameter value.** |